# Setup
## Imports

In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import matplotlib.pyplot as plt
import json
import pandas as pd
from typing import Optional, Sequence, Iterable, Literal

from generative_social_choice.utils.helper_functions import get_base_dir_path
from generative_social_choice.slates.voting_utils import gini
from generative_social_choice.slates.voting_algorithms import VotingAlgorithm
from generative_social_choice.ratings.utility_matrix import (
    extract_voter_utilities_from_info_csv,
    get_baseline_generate_slate_results,
)
from generative_social_choice.utils.postprocessing import (
    DEFAULT_COLORS,
    plot_sorted_utility_distributions,
    plot_likert_category_clustered_bar_chart,
    scalar_utility_metrics,
    plot_candidate_distribution_stacked,
    plot_sorted_utility_CIs,
    consolidate_duplicate_columns,
    bootstrap_df_rows,
    plot_scalar_clustered_confidence_intervals,
    compute_utilities_relative_to_reference,
)
from generative_social_choice.slates.voting_algorithms import (
    GreedyTotalUtilityMaximization,
    ExactTotalUtilityMaximization,
    ExactMinimumUtilityMaximization,
    LPTotalUtilityMaximization,
    ReweightedRangeVoting,
    SequentialPhragmenMinimax,
    ExactTotalUtilityMaximizationBruteSearch,
    StepwiseMonroe,
    ExactMonroe,
    GeometricTransformation,
)


## Load data

In [ ]:
from generative_social_choice.utils.helper_functions import get_results_paths

LABELLING_MODEL = "4o-mini"

our_pipeline_result_dirs = [get_results_paths(labelling_model=LABELLING_MODEL, baseline=False, run_id=s,  embedding_type="llm")['utility_matrix_file'] for s in range(10)]
our_pipeline_result_dirs

In [ ]:
baseline_result_dirs = [get_results_paths(labelling_model=LABELLING_MODEL, baseline=False, run_id=f"fish_{s}",  embedding_type="fish")['utility_matrix_file'] for s in range(10)]
baseline_result_dirs

In [ ]:
# Load csvs
from pathlib import Path

# The first 6 statements are seed statements, which are not part of the candidate set.
SEED_STATEMENT_IDS = [f"s{i}" for i in range(1, 7)]

def drop_seed_statements(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop(columns=[c for c in SEED_STATEMENT_IDS if c in df.columns])

# Load and merge our pipeline results
dfs_ours = []
for run_id, csv_path in enumerate(our_pipeline_result_dirs):
    df = pd.read_csv(csv_path, index_col=0, header=[0])
    df = drop_seed_statements(df)
    # Create multiindex columns: (run_id, statement_id)
    df.columns = pd.MultiIndex.from_product([[run_id], df.columns], names=['run_id', 'statement_id'])
    dfs_ours.append(df)

# Merge all our pipeline dataframes
df_ours = pd.concat(dfs_ours, axis=1)
df_ours

In [ ]:

# Load and merge baseline results
dfs_baseline = []
for run_id, csv_path in enumerate(baseline_result_dirs):
    df = pd.read_csv(csv_path, index_col=0, header=[0])
    df = drop_seed_statements(df)
    # Create multiindex columns: (run_id, statement_id)
    df.columns = pd.MultiIndex.from_product([[run_id], df.columns], names=['run_id', 'statement_id'])
    dfs_baseline.append(df)

# Merge all baseline dataframes
df_baseline = pd.concat(dfs_baseline, axis=1)

print(f"df_ours shape: {df_ours.shape}")
print(f"df_baseline shape: {df_baseline.shape}")

df_baseline

In [ ]:
# Fish et al.'s baseline pipeline (the "Baseline" series in new_pipeline_results.ipynb).
#
# NOTE: this is a pipeline output, not a voting algorithm. Every recorded baseline run chose a
# slate of exactly 5, and its voter ratings are only partially observed (~40% of the voter x
# candidate approval matrix is missing, because the pipeline is adaptive and never rates every
# voter against every candidate). So its assignment cannot be recomputed for another slate size
# offline -- that would require re-running generate_slate, which makes LLM calls. The baseline is
# therefore only shown in the slate-5 plots.
FISH_BASELINE_LABEL = "Baseline"
FISH_BASELINE_SLATE_SIZE = 5
# The baseline is not a voting algorithm, so it has no place in voting_algorithms_to_test.
# Instead it is drawn immediately after this algorithm.
FISH_BASELINE_AFTER = StepwiseMonroe(objective="total").display_name

fish_baseline_utilities, _ = get_baseline_generate_slate_results(
    run_ids=range(15), embedding_type="llm", candidate_pool="persistent"
)


def series_order(algs) -> list[str]:
    """Series order for the plots: the order of `algs`, with the baseline slotted in after
    FISH_BASELINE_AFTER. voting_algorithms_to_test is the single source of truth -- reorder
    it to reorder the plots.

    The baseline always gets a slot here, even for slate sizes where it is not plotted. That
    reserves its color, so the algorithms after it keep the same color at every slate size.
    Series that are not plotted are simply skipped by order_series.
    """
    names = [a.display_name if hasattr(a, "display_name") else str(a) for a in algs]
    if FISH_BASELINE_AFTER in names:
        at = names.index(FISH_BASELINE_AFTER) + 1
    else:
        print(f"warning: {FISH_BASELINE_AFTER!r} not plotted; putting baseline last")
        at = len(names)
    return names[:at] + [FISH_BASELINE_LABEL] + names[at:]


def append_fish_baseline(utilities: pd.DataFrame) -> pd.DataFrame:
    """Add the fish baseline as one more series of a (algorithm, run_id) utilities frame."""
    fish = fish_baseline_utilities.copy()
    fish.columns = pd.MultiIndex.from_tuples(
        [(FISH_BASELINE_LABEL, run_id) for run_id in fish.columns],
        names=utilities.columns.names,
    )
    # Rows are agent ids in both frames, so concat aligns on label.
    return pd.concat([utilities, fish], axis=1)


def order_series(utilities: pd.DataFrame, order: list[str]) -> pd.DataFrame:
    """Reorder the series of a (algorithm, run_id) utilities frame to follow `order`."""
    present = list(dict.fromkeys(utilities.columns.get_level_values(0)))
    unknown = [label for label in present if label not in order]
    if unknown:
        print(f"warning: series not in the plotting order, appended at end: {unknown}")
    ordered = [label for label in order if label in present] + unknown
    cols = [col for label in ordered for col in utilities.columns if col[0] == label]
    return utilities[cols]


def series_colors(utilities: pd.DataFrame, order: list[str]) -> list[str]:
    """Color for each plotted series, fixed by its slot in `order` rather than by its position
    in this particular plot. That way a series keeps its color across slate sizes even when
    another series (the baseline) is absent."""
    present = list(dict.fromkeys(utilities.columns.get_level_values(0)))
    return [
        DEFAULT_COLORS[order.index(label) % len(DEFAULT_COLORS)]
        if label in order
        else DEFAULT_COLORS[-1]
        for label in present
    ]


print(f"fish baseline utilities: {fish_baseline_utilities.shape} (slate size {FISH_BASELINE_SLATE_SIZE})")
fish_baseline_utilities.mean().round(4)

In [ ]:
# Helper Functions for Voting and Plotting

from generative_social_choice.slates.voting_utils import voter_utilities
from typing import Tuple, List, Any, Hashable, Iterable


def run_voting_algorithms(
    utility_df: pd.DataFrame,
    voting_algorithms: Tuple[Any, ...],
    slate_size: int = 5
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Run voting algorithms on a utility dataframe.
    
    Args:
        utility_df: DataFrame with MultiIndex columns (run_id, statement_id)
        voting_algorithms: Tuple of voting algorithm instances to test
        slate_size: Number of statements to select in each slate
        
    Returns:
        Tuple of (df_voter_utilities, df_voter_assignments, df_slates)
    """
    results_data = []
    
    # Iterate through each run in utility_df
    for run_id in utility_df.columns.get_level_values('run_id').unique():
        # Extract utility matrix for this run
        utility_matrix = utility_df[run_id]
        
        # Run each voting algorithm
        for voting_algorithm in voting_algorithms:
            # Run the voting algorithm
            slate, assignments = voting_algorithm.vote(
                rated_votes=utility_matrix.copy(),
                slate_size=slate_size
            )
            
            # Extract voter utilities from assignments
            voter_utilities_series = voter_utilities(
                rated_votes=utility_matrix,
                assignments_series=assignments['candidate_id'],
                output_column_name='utility'
            )
            
            # Store the result with run_id and algorithm name
            results_data.append({
                'run_id': run_id,
                'algorithm': voting_algorithm,
                'utilities': voter_utilities_series,
                'assignments': assignments['candidate_id'],
                'slate': sorted(slate)
            })
    
    # Create dataframe with multiindex columns for voter utilities
    dfs_utilities_list = []
    for result in results_data:
        df_temp = pd.DataFrame({
            (result['run_id'], result['algorithm']): result['utilities']
        })
        df_temp.columns = pd.MultiIndex.from_tuples(
            df_temp.columns, 
            names=['run_id', 'algorithm']
        )
        dfs_utilities_list.append(df_temp)
    
    df_voter_utilities = pd.concat(dfs_utilities_list, axis=1)
    
    # Create dataframe with multiindex columns for voter assignments
    dfs_assignments_list = []
    for result in results_data:
        df_temp = pd.DataFrame({
            (result['run_id'], result['algorithm']): result['assignments']
        })
        df_temp.columns = pd.MultiIndex.from_tuples(
            df_temp.columns, 
            names=['run_id', 'algorithm']
        )
        dfs_assignments_list.append(df_temp)
    
    df_voter_assignments = pd.concat(dfs_assignments_list, axis=1)
    
    # Create dataframe for slates
    slate_data = {}
    for result in results_data:
        col_key = (result['run_id'], result['algorithm'])
        slate_list = list(result['slate']) + [None] * (slate_size - len(result['slate']))
        slate_data[col_key] = slate_list[:slate_size]
    
    df_slates = pd.DataFrame(
        slate_data,
        index=range(slate_size)
    )
    df_slates.columns = pd.MultiIndex.from_tuples(
        df_slates.columns,
        names=['run_id', 'algorithm']
    )
    
    return df_voter_utilities, df_voter_assignments, df_slates


def reorganize_utilities_for_plotting(
    df_voter_utilities: pd.DataFrame
) -> pd.DataFrame:
    """
    Reorganize MultiIndex columns so algorithm is first level, run_id is second.
    
    Args:
        df_voter_utilities: DataFrame with columns (run_id, algorithm)
        
    Returns:
        DataFrame with columns (algorithm, run_id) using display names
    """
    utilities_swapped = df_voter_utilities.copy()
    # Swap levels so algorithm is first, run_id is second
    utilities_swapped.columns = utilities_swapped.columns.swaplevel(0, 1)
    # Replace the 'algorithm' part with display_name
    new_columns = pd.MultiIndex.from_tuples(
        [
            (alg.display_name if hasattr(alg, 'display_name') else str(alg), run_id)
            for alg, run_id in utilities_swapped.columns
        ],
        names=['algorithm', 'run_id']
    )
    utilities_swapped.columns = new_columns
    return utilities_swapped


def filter_utilities_for_plotting(
    utilities_swapped: pd.DataFrame,
    algos_to_plot: Tuple[Any, ...],
    run_ids_to_plot: Optional[List[int]] = None,
) -> pd.DataFrame:
    """
    Filter utilities dataframe by run_ids and algorithms.
    
    Args:
        utilities_swapped: DataFrame with columns (algorithm, run_id)
        run_ids_to_plot: List of run_ids to include
        algos_to_plot: Tuple of algorithm instances to include
        
    Returns:
        Filtered DataFrame
    """
    if run_ids_to_plot is None:
        run_ids_to_plot = utilities_swapped.columns.get_level_values('run_id').unique()

    algo_display_names = [
        algo.display_name if hasattr(algo, 'display_name') else str(algo)
        for algo in algos_to_plot
    ]
    
    if isinstance(utilities_swapped.columns, pd.MultiIndex):
        filtered_cols = [
            col for col in utilities_swapped.columns
            if col[1] in run_ids_to_plot and col[0] in algo_display_names
        ]
    else:
        filtered_cols = [
            col for col in utilities_swapped.columns
            if col in algo_display_names
        ]
    
    return utilities_swapped[filtered_cols]

def create_all_plots(
    utilities_for_line_plot: pd.DataFrame,
    dataset_name: str = "",
    show_titles: bool = True,
    which_plots: Iterable[int] = {1, 2, 3, 4},
    plot_series_order: Optional[List[str]] = None,
) -> list:
    """
    Create all plots for a given utilities dataframe.
    
    Args:
        utilities_for_line_plot: Filtered utilities DataFrame
        dataset_name: Optional name to include in plot titles
    """
    return_figs = []
    # Plot 1: Sorted utility distributions
    if 1 in which_plots:
        fig = plot_sorted_utility_distributions(
            utilities_for_line_plot, 
            figsize=(4.5, 5.5)
        )
        ax = plt.gca()
        ax.set_xlim([70, 100])
        if dataset_name and show_titles:
            ax.set_title(f"{dataset_name}: Sorted Utility Distributions")
        return_figs.append(fig)

    # Plot 2: Sorted utility CIs (without confidence intervals)
    if 2 in which_plots:
        figs = plot_sorted_utility_CIs(
            utilities_for_line_plot, 
            do_CI=False, 
            figsize=(5.5, 4)
        )
        if dataset_name and show_titles:
            if isinstance(figs, (list, tuple)):
                for f in figs:
                    f.suptitle(f"{dataset_name}: Sorted Utility CIs")
            else:
                figs.suptitle(f"{dataset_name}: Sorted Utility CIs")
        return_figs.extend(figs)
    # Plot 3: Utilities relative to Exact
    if 3 in which_plots:
        sorted_utilities_relative_to_exact = compute_utilities_relative_to_reference(
            utilities_for_line_plot
        )
        fig1 = plot_sorted_utility_CIs(
            sorted_utilities_relative_to_exact, 
            figsize=(4.5, 4.5), 
            confidence_level=0.95, 
            do_CI=False, 
            do_sort=False,
            ylabel="Utility difference from Exact",
            colors=DEFAULT_COLORS[1:]
        )
        ax1 = plt.gca()
        if dataset_name and show_titles:
            ax1.set_title(f"{dataset_name}: Utilities Relative to Exact")
        return_figs.append(fig1)
    # Plot 4: Scalar utility metrics with confidence intervals
    if 4 in which_plots:
        # Calculate scalar metrics (columns are scenarios, rows are metrics)
        # scalar_metrics = scalar_utility_metrics(utilities_for_line_plot, metrics=("Mean Log", "Gini"))
        scalar_metrics = scalar_utility_metrics(utilities_for_line_plot, metrics=("Mean", "Mean of Bottom 50%", "Minimum", "Gini", "Mean Log"))
        # Transpose so rows have (algorithm, run_id) MultiIndex and columns are metrics
        # scalar_metrics_transposed = scalar_metrics.T
        # Calculate confidence intervals (groups by algorithm, treats run_id as sample)
    # Output: Rows (algorithm, statistic), Columns (metrics)
        # sort_groups=False keeps the series in column order, which order_series has already
        # set to SERIES_ORDER. With the default (sorted) grouping they would come out alphabetical.
        scalar_confidence_intervals = bootstrap_df_rows(
            scalar_metrics,
            confidence_level=0.95,
            n_bootstrap=400,
            seed=1612,
            sort_groups=False,
        )
        # Restructure for plotting: need Rows (metric, statistic), Columns (algorithm)
        # First transpose to get Rows (metrics), Columns (algorithm, statistic)
        # Then stack to get Rows (metric, algorithm, statistic)
        # Then unstack algorithm to get Rows (metric, statistic), Columns (algorithm)
        # scalar_ci_for_plot = scalar_confidence_intervals.T.stack(level=0).unstack(level=0)
        # Separate Gini from other metrics for secondary axis
        # Minimum is far below the other utility metrics, so it gets its own subplot. Otherwise the shared
        # y-axis would have to span it, squashing the differences between algorithms on the others.
        utility_metrics = scalar_confidence_intervals.drop(columns=['Minimum', 'Gini', 'Mean Log'])
        min_metric = scalar_confidence_intervals[['Minimum']]
        gini_metric = scalar_confidence_intervals[['Gini']]
        log_metric = scalar_confidence_intervals[['Mean Log']]
        # Each subplot's y-axis fits its own data
        # bar_index_level=0 means metrics (first level) are bar labels, algorithms are clusters
        # Colors are keyed to the plotting order, so a series keeps its color across slate
        # sizes even when another series (Baseline) is absent.
        fig2 = plot_scalar_clustered_confidence_intervals(
            utility_metrics,
            colors=series_colors(utilities_for_line_plot, plot_series_order) if plot_series_order else None,
            bar_index='mean',
            error_bar_lower_index='lower bound',
            error_bar_upper_index='upper bound',
            bar_index_level=0,
            y_label="Utility",
            # Left of all subplots, so the legend never covers the data
            legend_loc="outside left center",
            legend_frameon=False,
            fig_size=(15.84, 6.5),
            # Narrow gaps between clusters leave room to spread out the series within each cluster
            cluster_width=1.15,
            cluster_gap=0.7,
            secondary_axis_df=min_metric,
            secondary_y_label="Utility",
            tertiary_axis_df=log_metric,
            tertiary_y_label="log(Utility)",
            quaternary_axis_df=gini_metric,
            quaternary_y_label="Gini",
            color_by_position=True,
        )
        if dataset_name and show_titles:
            fig2.suptitle(f"{dataset_name}: Scalar Utility Metrics")
        return_figs.append(fig2)
    
    return return_figs
        

In [ ]:
UtilityMatrixSource = Literal["ours", "baseline", "concat"]
ALL_UTILITY_MATRICES: tuple[UtilityMatrixSource, ...] = ("ours", "baseline", "concat")


def _validate_utility_matrices(utility_matrices: Sequence[UtilityMatrixSource]) -> None:
    invalid = set(utility_matrices) - set(ALL_UTILITY_MATRICES)
    assert not invalid, f"Unknown utility matrices {invalid}, expected a subset of {ALL_UTILITY_MATRICES}"


def vote_baseline_and_ours(
    algs: Sequence[VotingAlgorithm],
    our_utility_df: pd.DataFrame,
    baseline_utility_df: pd.DataFrame,
    slate_size: int = 5,
    utility_matrices: Sequence[UtilityMatrixSource] = ALL_UTILITY_MATRICES,
):
    """Columns of the returned utilities are (run_id, algorithm instance). Algorithms are converted to
    their display names only at plot time, so renaming an algorithm doesn't require re-running the vote.

    `utility_matrices` selects which utility matrices to vote on. "concat" needs the votes of both "ours" and
    "baseline", so it votes on both. The utilities of a matrix that isn't voted on are returned as None."""
    _validate_utility_matrices(utility_matrices)
    df_voter_utilities_ours = None
    if "ours" in utility_matrices or "concat" in utility_matrices:
        df_voter_utilities_ours, df_voter_assignments_ours, df_slates_ours = run_voting_algorithms(
            utility_df=our_utility_df,
            voting_algorithms=algs,
            slate_size=slate_size
        )

    df_voter_utilities_baseline = None
    if "baseline" in utility_matrices or "concat" in utility_matrices:
        df_voter_utilities_baseline, df_voter_assignments_baseline, df_slates_baseline = run_voting_algorithms(
            utility_df=baseline_utility_df,
            voting_algorithms=algs,
            slate_size=slate_size
        )

    return df_voter_utilities_ours, df_voter_utilities_baseline

def create_all_plots_from_utilities(
    utilities_ours: Optional[pd.DataFrame],
    utilities_baseline: Optional[pd.DataFrame],
    algs: Sequence[VotingAlgorithm],
    which_plots: Iterable[int] = {3, 4},
    show_titles: bool = True,
    include_fish_baseline: bool = False,
    which_matrices: Sequence[UtilityMatrixSource] = ALL_UTILITY_MATRICES,
):
    """`utilities_ours` and `utilities_baseline` are the outputs of vote_baseline_and_ours.

    `utility_matrices` selects which utility matrices to plot, in the order ours, baseline, concat.
    "concat" needs both `utilities_ours` and `utilities_baseline`."""
    _validate_utility_matrices(which_matrices)
    needs_ours = "ours" in which_matrices or "concat" in which_matrices
    needs_baseline = "baseline" in which_matrices or "concat" in which_matrices
    assert not needs_ours or utilities_ours is not None, \
        "utilities_ours is None; vote on it with utility_matrices including 'ours' or 'concat'"
    assert not needs_baseline or utilities_baseline is not None, \
        "utilities_baseline is None; vote on it with utility_matrices including 'baseline' or 'concat'"

    def prepare(utilities: pd.DataFrame) -> pd.DataFrame:
        return filter_utilities_for_plotting(
            reorganize_utilities_for_plotting(utilities),
            list(algs),
            None,
        )

    # Series order (and therefore color) comes from the order of `algs`. The baseline keeps its
    # slot even when it is not plotted, so colors do not shift between slate sizes.
    order = series_order(algs)

    def finalize(utilities_for_line_plot: pd.DataFrame) -> pd.DataFrame:
        # The fish baseline only exists at slate size 5, so it is opt-in per slate size.
        if include_fish_baseline:
            utilities_for_line_plot = append_fish_baseline(utilities_for_line_plot)
        return order_series(utilities_for_line_plot, order)

    utilities_for_line_plot_ours = prepare(utilities_ours) if needs_ours else None
    utilities_for_line_plot_baseline = prepare(utilities_baseline) if needs_baseline else None

    plots = []
    if "ours" in which_matrices:
        plots += create_all_plots(finalize(utilities_for_line_plot_ours), dataset_name="Ours", show_titles=show_titles, which_plots=which_plots, plot_series_order=order)
    if "baseline" in which_matrices:
        plots += create_all_plots(finalize(utilities_for_line_plot_baseline), dataset_name="Baseline", show_titles=show_titles, which_plots=which_plots, plot_series_order=order)
    if "concat" not in which_matrices:
        return tuple(plots)

    baseline_run_ids = utilities_for_line_plot_baseline.columns.get_level_values('run_id').unique()
    max_baseline_run_id = max(baseline_run_ids) if len(baseline_run_ids) > 0 else -1
    # Create a copy of the "ours" data with offset run IDs
    utilities_ours_offset = utilities_for_line_plot_ours.copy()
    if isinstance(utilities_ours_offset.columns, pd.MultiIndex):
        # Get the current run_ids and offset them
        offset = max_baseline_run_id + 1
        new_columns = []
        for col in utilities_ours_offset.columns:
            algorithm, run_id = col
            new_run_id = run_id + offset
            new_columns.append((algorithm, new_run_id))
        utilities_ours_offset.columns = pd.MultiIndex.from_tuples(
            new_columns,
            names=utilities_ours_offset.columns.names
        )

    # Concatenate the offset "ours" data with baseline data
    utilities_for_line_plot_combined = pd.concat(
        [utilities_ours_offset, utilities_for_line_plot_baseline],
        axis=1
    )
    plots += create_all_plots(finalize(utilities_for_line_plot_combined), dataset_name="Combined", show_titles=show_titles, which_plots=which_plots, plot_series_order=order)

    return tuple(plots)

# Script

In [ ]:
# The order of this tuple is the order the series are plotted in, and it fixes each series'
# color. Reorder here to reorder the plots.
voting_algorithms_to_test = (
    ExactTotalUtilityMaximization(),
    ExactTotalUtilityMaximization(utility_transform=GeometricTransformation(p=1.5)),
    ExactTotalUtilityMaximization(utility_transform=GeometricTransformation(p=6.0)),
    # ExactMinimumUtilityMaximization(),
    GreedyTotalUtilityMaximization(),
    LPTotalUtilityMaximization(),
    ExactMonroe(),
    StepwiseMonroe(objective="total"),
    StepwiseMonroe(final_assignment="greedy_equal"),
    StepwiseMonroe(final_assignment="optimal_equal"),
    StepwiseMonroe(final_assignment="free"),
    # ExactMonroe(objective="minimum"),
    # ReweightedRangeVoting(),
    # ReweightedRangeVoting(k=0.5),
    # SequentialPhragmenMinimax(),
)

# Plots

In [ ]:
ALGOS_TO_PLOT = list(voting_algorithms_to_test)
# ALGOS_TO_PLOT.remove(ExactTotalUtilityMaximizationBruteSearch(utility_transform=GeometricTransformation(p=6.0)))
# ALGOS_TO_PLOT.remove(ExactTotalUtilityMaximization(utility_transform=GeometricTransformation(p=6.0)))
# ALGOS_TO_PLOT.remove(ReweightedRangeVoting())
WHICH_PLOTS = {4,}
# Which utility matrices to vote on and plot: any of "ours", "baseline", "concat"
UTILITY_MATRICES = ("ours",)

## Slate 3

In [ ]:
# slate3_utilities_ours, slate3_utilities_baseline = vote_baseline_and_ours(
#     voting_algorithms_to_test,
#     df_ours,
#     df_baseline,
#     slate_size=3,
#     utility_matrices=UTILITY_MATRICES,
# )

In [ ]:
# figs = create_all_plots_from_utilities(slate3_utilities_ours, slate3_utilities_baseline, algs=ALGOS_TO_PLOT, which_plots=WHICH_PLOTS, which_matrices=UTILITY_MATRICES, show_titles=False)

## Slate 5

In [ ]:
slate5_utilities_ours, slate5_utilities_baseline = vote_baseline_and_ours(
    voting_algorithms_to_test,
    df_ours,
    df_baseline,
    slate_size=5,
    utility_matrices=UTILITY_MATRICES,
)

In [ ]:
create_all_plots_from_utilities(slate5_utilities_ours, slate5_utilities_baseline, algs=ALGOS_TO_PLOT, which_plots=WHICH_PLOTS, which_matrices=UTILITY_MATRICES, include_fish_baseline=True, show_titles=False)


## Slate 8

In [ ]:
slate8_utilities_ours, slate8_utilities_baseline = vote_baseline_and_ours(
    voting_algorithms_to_test,
    df_ours,
    df_baseline,
    slate_size=8,
    utility_matrices=UTILITY_MATRICES,
)

In [ ]:
create_all_plots_from_utilities(slate8_utilities_ours, slate8_utilities_baseline, algs=ALGOS_TO_PLOT, which_plots=WHICH_PLOTS, which_matrices=UTILITY_MATRICES)

